In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# 1. Define GridWorld Parameters

GRID_SIZE = 4

# Terminal states
TERMINAL_STATES = [(0, 0), (3, 3)]

# Actions
ACTIONS = ['U', 'D', 'L', 'R']

# Reward for each movement
STEP_REWARD = -1

# Discount factor
GAMMA = 0.9

# Convergence threshold
THETA = 1e-6

In [ ]:
# 2. Stochastic Transition Probabilities

# Intended action  : 0.8
# Perpendicular actions : 0.1 each
#
# Example:
# If the agent chooses UP:
# UP    -> 0.8
# LEFT  -> 0.1
# RIGHT -> 0.1
#
# The agent may therefore slip to another direction.

TRANSITION_PROBS = {
    'U': [('U', 0.8), ('L', 0.1), ('R', 0.1)],
    'D': [('D', 0.8), ('L', 0.1), ('R', 0.1)],
    'L': [('L', 0.8), ('U', 0.1), ('D', 0.1)],
    'R': [('R', 0.8), ('U', 0.1), ('D', 0.1)]
}

In [ ]:
# 3. Helper Functions


def is_terminal(state):
    """Check whether a state is terminal."""
    return state in TERMINAL_STATES


def move(state, action):
    """
    Move agent according to the selected action.

    If the movement takes the agent outside the grid,
    the agent remains in the same state.
    """

    row, col = state

    if action == 'U':
        row -= 1
    elif action == 'D':
        row += 1
    elif action == 'L':
        col -= 1
    elif action == 'R':
        col += 1

    # Boundary condition
    if row < 0 or row >= GRID_SIZE or col < 0 or col >= GRID_SIZE:
        return state

    return (row, col)


In [ ]:
# 4. Generate All States


states = []

for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):
        states.append((row, col))

# 5. Initialize Value Function

V = np.zeros((GRID_SIZE, GRID_SIZE))

# Terminal states have value 0
for state in TERMINAL_STATES:
    V[state] = 0


In [ ]:
# 6. Value Iteration

iteration = 0

while True:

    delta = 0

    # Create a copy so that all updates use the previous values
    new_V = V.copy()

    for state in states:

        # Terminal states are not updated
        if is_terminal(state):
            continue

        action_values = []

        # Calculate value for every action
        for action in ACTIONS:

            expected_value = 0

            # Consider stochastic transitions
            for actual_action, probability in TRANSITION_PROBS[action]:

                next_state = move(state, actual_action)

                reward = STEP_REWARD

                # Bellman expectation equation
                expected_value += probability * (
                    reward + GAMMA * V[next_state]
                )

            action_values.append(expected_value)

        # Bellman optimality equation
        new_V[state] = max(action_values)

        delta = max(
            delta,
            abs(new_V[state] - V[state])
        )

    V = new_V

    iteration += 1

    # Check convergence
    if delta < THETA:
        break



In [ ]:
# 7. Calculate Optimal Policy

policy = np.empty((GRID_SIZE, GRID_SIZE), dtype=object)

for state in states:

    if is_terminal(state):
        policy[state] = 'T'
        continue

    action_values = {}

    for action in ACTIONS:

        expected_value = 0

        for actual_action, probability in TRANSITION_PROBS[action]:

            next_state = move(state, actual_action)

            reward = STEP_REWARD

            expected_value += probability * (
                reward + GAMMA * V[next_state]
            )

        action_values[action] = expected_value

    # Select action with maximum expected value
    best_action = max(
        action_values,
        key=action_values.get
    )

    policy[state] = best_action


In [ ]:
# 8. Print Number of Iterations

print("=" * 60)
print("VALUE ITERATION - GRIDWORLD")
print("=" * 60)

print("\nNumber of iterations required for convergence:",
      iteration)


# 9. Print State Value Function

print("\nState Value Function:")
print("-" * 40)

for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):
        print(
            f"V({row},{col}) = {V[row, col]:8.4f}",
            end="    "
        )
    print()


In [ ]:
# 10. Print Optimal Policy

print("\nOptimal Policy:")
print("-" * 40)

for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):
        print(
            f"{policy[row, col]:^5}",
            end=" "
        )
    print()



In [ ]:
# 11. Display Value Function as Heatmap

plt.figure(figsize=(8, 6))

sns.heatmap(
    V,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    linewidths=1,
    square=True,
    cbar=True
)

plt.title("Value Function after Value Iteration")
plt.xlabel("Column")
plt.ylabel("Row")

plt.show()



In [ ]:
# 12. Display Optimal Policy

plt.figure(figsize=(8, 6))

# Background grid
plt.imshow(
    np.zeros((GRID_SIZE, GRID_SIZE)),
    cmap="Greys",
    alpha=0.2
)

# Display policy arrows
arrow_map = {
    'U': '↑',
    'D': '↓',
    'L': '←',
    'R': '→',
    'T': 'T'
}

for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):

        action = policy[row, col]

        plt.text(
            col,
            row,
            arrow_map[action],
            ha='center',
            va='center',
            fontsize=24
        )

# Grid formatting
plt.xticks(range(GRID_SIZE))
plt.yticks(range(GRID_SIZE))

plt.grid(
    True,
    linewidth=1
)

plt.title("Optimal Policy - Stochastic GridWorld")
plt.xlabel("Column")
plt.ylabel("Row")

plt.show()



In [ ]:
# 13. Display GridWorld States

plt.figure(figsize=(8, 6))

plt.imshow(
    np.zeros((GRID_SIZE, GRID_SIZE)),
    cmap="Greys",
    alpha=0.15
)

for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):

        state = (row, col)

        if state == (0, 0):
            text = "START\nTerminal"

        elif state == (3, 3):
            text = "GOAL\nTerminal"

        else:
            text = f"({row},{col})"

        plt.text(
            col,
            row,
            text,
            ha='center',
            va='center',
            fontsize=12
        )

plt.xticks(range(GRID_SIZE))
plt.yticks(range(GRID_SIZE))

plt.grid(
    True,
    linewidth=1
)

plt.title("4 × 4 GridWorld Environment")
plt.xlabel("Column")
plt.ylabel("Row")

plt.show()



In [ ]:
# 14. Display Transition Probabilities

print("\nStochastic Transition Probabilities:")
print("-" * 50)

for action in ACTIONS:

    print(f"\nChosen Action: {action}")

    for actual_action, probability in TRANSITION_PROBS[action]:

        print(
            f"  Actual Action = {actual_action}, "
            f"Probability = {probability}"
        )


print("\n" + "=" * 60)
print("Experiment completed successfully.")
print("=" * 60)